# 01 — Own test set (2000 images)

Что делает ноутбук:
1. скачивает **1000 real** фото из интернета (Unsplash Lite или COCO val2017) вместе с их описаниями;
2. генерирует **600 fake (t2i)** в Stable Diffusion по описаниям этих же реальных фото (чтобы темы совпадали);
3. делает **400 fake (i2i)** — переделывает 400 реальных фото через img2img, запоминая id исходника;
4. приводит всё к единому формату (JPEG, 512 по длинной стороне) и пишет `own/metadata.csv`
   в той же схеме, что и у выборки из Community Forensics (ноутбук 02).

**Перед запуском:** Runtime → Change runtime type → **GPU (T4)**. Запускать ячейки сверху вниз.
Результат: папка `/content/own` и архив `own.zip` (при желании копируется в Google Drive).

In [ ]:
!pip -q install "diffusers>=0.27" transformers accelerate pillow pandas requests tqdm

In [ ]:
import os, io, csv, json, random, zipfile, shutil
import requests, pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

CFG = dict(
    REAL_SOURCE = "coco",       # "coco" (COCO val2017 — качается напрямую, надёжно) или "unsplash" (Unsplash Lite; сайт может отдавать 403)
    N_REAL = 1000,
    N_T2I  = 600,
    N_I2I  = 400,
    MODEL_ID = "stabilityai/sdxl-turbo",   # быстрый, помещается на T4; альтернатива: "stable-diffusion-v1-5/stable-diffusion-v1-5"
    GEN_NAME = "sdxl-turbo",               # как генератор будет называться в metadata
    MAX_SIDE = 512,      # длинная сторона после нормализации
    JPEG_Q   = 95,       # качество JPEG для ВСЕХ картинок (и real, и fake)
    SEED = 42,
    OUT = "/content/own",
    SAVE_TO_DRIVE = True,
)
random.seed(CFG["SEED"])

IMG_DIR  = os.path.join(CFG["OUT"], "images")
ORIG_DIR = os.path.join(CFG["OUT"], "originals")
WORK     = "/content/work"
for d in (IMG_DIR, ORIG_DIR, WORK):
    os.makedirs(d, exist_ok=True)

# Единая схема metadata — ровно такая же в ноутбуке 02 для Community Forensics
COLUMNS = ["id", "filename", "label", "origin", "source", "fake_type", "generator",
           "prompt", "caption", "original_id", "category", "source_url", "held_out", "split"]
rows = []           # сюда складываем строки metadata
_counter = [0]

def next_id():
    _counter[0] += 1
    return f"own_{_counter[0]:06d}"

def normalize(img: Image.Image) -> Image.Image:
    """Единый формат: RGB, EXIF-поворот применён, длинная сторона = MAX_SIDE."""
    img = ImageOps.exif_transpose(img).convert("RGB")
    w, h = img.size
    s = CFG["MAX_SIDE"] / max(w, h)
    if s < 1:
        img = img.resize((round(w * s), round(h * s)), Image.LANCZOS)
    return img

def save_norm(img: Image.Image, img_id: str) -> str:
    fn = f"{img_id}.jpg"
    normalize(img).save(os.path.join(IMG_DIR, fn), "JPEG", quality=CFG["JPEG_Q"], optimize=True)
    return fn

def add_row(img_id, fn, label, source, fake_type="none", generator="", prompt="", caption="",
            original_id="", category="", source_url=""):
    rows.append(dict(id=img_id, filename=fn, label=label, origin="own", source=source,
                     fake_type=fake_type, generator=generator, prompt=prompt, caption=caption,
                     original_id=original_id, category=category, source_url=source_url,
                     held_out="", split="test_own"))

def download(url, path, chunk=1 << 20):
    if os.path.exists(path):
        return path
    headers = {"User-Agent": "Mozilla/5.0 (research dataset download)"}
    with requests.get(url, stream=True, timeout=120, headers=headers, allow_redirects=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=os.path.basename(path)) as bar:
            for c in r.iter_content(chunk):
                f.write(c); bar.update(len(c))
    return path

print("OK, папки созданы:", CFG["OUT"])

## 1. Real — 1000 фото из интернета

По умолчанию — COCO val2017: 5000 обычных фотографий с Flickr с подписями, качается напрямую одним архивом.
Описание каждой фотографии сохраняем в `caption` — по нему дальше генерируются фейки на ту же тему.

In [ ]:
real_items = []   # (img_id, filename, caption, source_url) — нужно для шагов 2 и 3
source = CFG["REAL_SOURCE"]

if source == "unsplash":
    # Unsplash Lite Dataset: ~25k фото, официальный бесплатный набор с TSV-таблицей ссылок и описаний.
    # Прямая ссылка иногда отдаёт 403 — тогда либо скачайте архив вручную с https://unsplash.com/data
    # и положите его в /content/work/unsplash_lite.zip, либо ноутбук сам переключится на COCO.
    try:
        zpath = download("https://unsplash.com/data/lite/latest", f"{WORK}/unsplash_lite.zip")
    except Exception as e:
        print("Unsplash недоступен:", e, "\n→ переключаюсь на COCO val2017")
        source = "coco"

if source == "unsplash":
    with zipfile.ZipFile(zpath) as z:
        parts = [n for n in z.namelist() if os.path.basename(n).startswith("photos.tsv")]
        photos = pd.concat([pd.read_csv(z.open(n), sep="\t", low_memory=False) for n in parts], ignore_index=True)
    photos["caption"] = photos["photo_description"].fillna(photos.get("ai_description"))
    photos = photos.dropna(subset=["caption", "photo_image_url"])
    photos = photos[photos["caption"].str.len().between(15, 200)]
    cand = photos.sample(frac=1, random_state=CFG["SEED"]).reset_index(drop=True)

    for _, p in tqdm(cand.iterrows(), total=len(cand), desc="unsplash"):
        if len(real_items) >= CFG["N_REAL"]:
            break
        try:
            url = p["photo_image_url"] + "?w=1024&fm=jpg"
            r = requests.get(url, timeout=30); r.raise_for_status()
            img = Image.open(io.BytesIO(r.content))
            img_id = next_id()
            img.save(os.path.join(ORIG_DIR, f"{img_id}_orig.jpg"), "JPEG", quality=100)
            fn = save_norm(img, img_id)
            add_row(img_id, fn, "real", "unsplash_lite", caption=str(p["caption"]).strip(), source_url=p["photo_url"])
            real_items.append((img_id, fn, str(p["caption"]).strip(), p["photo_url"]))
        except Exception as e:
            print("skip:", e)

elif source == "coco":
    # COCO val2017: 5000 фото с Flickr + подписи людей (архив ~780 МБ + аннотации ~240 МБ, качается напрямую)
    download("http://images.cocodataset.org/zips/val2017.zip", f"{WORK}/val2017.zip")
    download("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", f"{WORK}/ann.zip")
    with zipfile.ZipFile(f"{WORK}/ann.zip") as z:
        caps = json.load(z.open("annotations/captions_val2017.json"))
    cap_by_img = {}
    for a in caps["annotations"]:
        cap_by_img.setdefault(a["image_id"], a["caption"].strip())
    file_by_id = {im["id"]: im["file_name"] for im in caps["images"]}
    ids = [i for i in file_by_id if i in cap_by_img]
    random.shuffle(ids)
    with zipfile.ZipFile(f"{WORK}/val2017.zip") as z:
        for coco_id in tqdm(ids[: CFG["N_REAL"]], desc="coco"):
            img = Image.open(z.open(f"val2017/{file_by_id[coco_id]}"))
            img_id = next_id()
            img.save(os.path.join(ORIG_DIR, f"{img_id}_orig.jpg"), "JPEG", quality=100)
            fn = save_norm(img, img_id)
            url = f"http://images.cocodataset.org/val2017/{file_by_id[coco_id]}"
            add_row(img_id, fn, "real", "coco_val2017", caption=cap_by_img[coco_id], source_url=url)
            real_items.append((img_id, fn, cap_by_img[coco_id], url))
else:
    raise ValueError("REAL_SOURCE должен быть 'unsplash' или 'coco'")

print(f"real: {len(real_items)} шт.")
pd.DataFrame(rows, columns=COLUMNS).to_csv(os.path.join(CFG["OUT"], "metadata.csv"), index=False)  # промежуточное сохранение

## 2. Fake t2i — 600 сгенерированных по описаниям реальных фото

Берём 600 случайных описаний из real-части и генерируем по ним. Так темы real и fake совпадают,
и детектор вынужден учиться на артефактах генерации, а не на сюжете.
`sdxl-turbo` делает картинку за 4 шага без guidance — на T4 ~1–2 с на изображение.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image

pipe = AutoPipelineForText2Image.from_pretrained(CFG["MODEL_ID"], torch_dtype=torch.float16, variant="fp16").to("cuda")
pipe.set_progress_bar_config(disable=True)
is_turbo = "turbo" in CFG["MODEL_ID"]
T2I_KW = dict(num_inference_steps=4, guidance_scale=0.0) if is_turbo else dict(num_inference_steps=30, guidance_scale=7.0)
PROMPT_TMPL = "{caption}, realistic photo"

t2i_src = random.sample(real_items, CFG["N_T2I"])
for (src_id, _, caption, _) in tqdm(t2i_src, desc="t2i"):
    prompt = PROMPT_TMPL.format(caption=caption)
    g = torch.Generator("cuda").manual_seed(random.randint(0, 2**31))
    img = pipe(prompt=prompt, width=512, height=512, generator=g, **T2I_KW).images[0]
    img_id = next_id()
    img.save(os.path.join(ORIG_DIR, f"{img_id}_orig.png"))     # оригинал генератора (PNG) — только для архива
    fn = save_norm(img, img_id)                                 # в датасет идёт JPEG, как и real
    add_row(img_id, fn, "fake", CFG["GEN_NAME"], fake_type="t2i", generator=CFG["GEN_NAME"],
            prompt=prompt, caption=caption, original_id=src_id)   # original_id = чьё описание взяли

print("t2i готово")
pd.DataFrame(rows, columns=COLUMNS).to_csv(os.path.join(CFG["OUT"], "metadata.csv"), index=False)

## 3. Fake i2i — 400 переделанных реальных фото

`strength=0.5`: картинка остаётся узнаваемой, но перерисована моделью. Это самый «трудный» подтип для детектора.
Берём реальные фото, которые НЕ использовались как источник описаний в шаге 2, чтобы пары не пересекались.

In [ ]:
pipe_i2i = AutoPipelineForImage2Image.from_pipe(pipe)   # та же модель, память не дублируется
pipe_i2i.set_progress_bar_config(disable=True)
I2I_KW = dict(num_inference_steps=4, strength=0.5, guidance_scale=0.0) if is_turbo else dict(num_inference_steps=30, strength=0.5, guidance_scale=7.0)

used = {s[0] for s in t2i_src}
pool = [r for r in real_items if r[0] not in used] or real_items
i2i_src = random.sample(pool, min(CFG["N_I2I"], len(pool)))

for (src_id, src_fn, caption, _) in tqdm(i2i_src, desc="i2i"):
    init = Image.open(os.path.join(IMG_DIR, src_fn)).convert("RGB").resize((512, 512))
    g = torch.Generator("cuda").manual_seed(random.randint(0, 2**31))
    img = pipe_i2i(prompt=PROMPT_TMPL.format(caption=caption), image=init, generator=g, **I2I_KW).images[0]
    img_id = next_id()
    img.save(os.path.join(ORIG_DIR, f"{img_id}_orig.png"))
    fn = save_norm(img, img_id)
    add_row(img_id, fn, "fake", CFG["GEN_NAME"], fake_type="i2i", generator=CFG["GEN_NAME"],
            prompt=PROMPT_TMPL.format(caption=caption), caption=caption, original_id=src_id)

print("i2i готово")

## 4. metadata.csv, проверка и архив

Сюда же можно добавить картинки из Bing / Gemini и т.п.: положите их в `/content/own/extra/<имя_генератора>/`
и запустите эту ячейку ещё раз — они будут нормализованы и добавлены в metadata с `fake_type=t2i`.

In [ ]:
EXTRA = os.path.join(CFG["OUT"], "extra")
if os.path.isdir(EXTRA):
    for gen in sorted(os.listdir(EXTRA)):
        gdir = os.path.join(EXTRA, gen)
        if not os.path.isdir(gdir):
            continue
        for f in sorted(os.listdir(gdir)):
            if not f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                continue
            img_id = next_id()
            fn = save_norm(Image.open(os.path.join(gdir, f)), img_id)
            add_row(img_id, fn, "fake", gen, fake_type="t2i", generator=gen, prompt="(manual)", category="")
            print("extra:", gen, f, "->", fn)

meta = pd.DataFrame(rows, columns=COLUMNS)
meta.to_csv(os.path.join(CFG["OUT"], "metadata.csv"), index=False)

print(meta.groupby(["label", "fake_type", "generator"]).size())
print("\nвсего:", len(meta), "| файлов в images/:", len(os.listdir(IMG_DIR)))
assert meta["filename"].is_unique and len(meta) == len(os.listdir(IMG_DIR))

# архив (без originals — они большие и для обучения не нужны)
shutil.make_archive("/content/own", "zip", root_dir=CFG["OUT"], base_dir=".")
print("архив: /content/own.zip")

if CFG["SAVE_TO_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/deepfake_paper", exist_ok=True)
    shutil.copy("/content/own.zip", "/content/drive/MyDrive/deepfake_paper/own.zip")
    print("скопировано в Google Drive: deepfake_paper/own.zip")

## 5. Быстрый взгляд на результат

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for ax_row, kind in zip(axes, [("real", "none"), ("fake", "t2i"), ("fake", "i2i")]):
    sub = meta[(meta.label == kind[0]) & (meta.fake_type == kind[1])].head(6)
    for ax, (_, r) in zip(ax_row, sub.iterrows()):
        ax.imshow(Image.open(os.path.join(IMG_DIR, r.filename))); ax.axis("off")
        ax.set_title(f"{r.label}/{r.fake_type}", fontsize=9)
plt.tight_layout(); plt.show()